## Example for extracting data for GPT prompting

### This is not the final/complete code but more about how to get the desired data from the table

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import json

## andmetabelid

In [2]:
filename = "/home/kaire/kaire/development/syntax_experiments/semantic_labelling/v04_verb-case_pattern/drive_data/v33_koondkorpus_transaktsioonid.db"
conn = sqlite3.connect(filename)
cursor = conn.cursor()

## graafiku punktide info

In [3]:
query = """SELECT verb, verb_compound, morph_case, log2_tag, unique_lemmas, log2_unique_lemmas, level, not_ann_words, ann_words, ann_unique_lemmas, olulisus
            FROM lines_class_info3
            """

class_info = pd.read_sql(query, conn)
class_info

,verb,verb_compound,morph_case,log2_tag,unique_lemmas,log2_unique_lemmas,level,not_ann_words,ann_words,ann_unique_lemmas,olulisus
0,aasima,,ad,-9.965784,1,0.000000,-,7.0,NaN,NaN,-
1,abistama,,abl,-9.965784,1,0.000000,-,6.0,NaN,NaN,-
2,abistama,,all,-9.965784,1,0.000000,-,15.0,NaN,NaN,-
3,adresseerima,,in,-9.965784,1,0.000000,-,6.0,NaN,NaN,-
4,aeglustama,,all,-9.965784,1,0.000000,-,8.0,NaN,NaN,-
...,...,...,...,...,...,...,...,...,...,...,...
21264,õnnestuma,,in,1.227736,312,8.285402,-,1286.0,541.0,209.0,-
21265,õppima,,in,4.905379,547,9.095397,n90,5848.0,5724.0,465.0,0.0
21266,ütlema,,ad,-5.088166,459,8.842350,n10,9790.0,362.0,112.0,0.0
21267,ütlema,,el,0.974385,629,9.296916,n70,2839.0,949.0,337.0,0.97997


## võtta ainult n80 tsooni punktid

In [4]:
filtered_class = class_info[class_info["level"]=="n80"]
filtered_class = filtered_class.sort_values(["olulisus"])

In [5]:
filtered_class

,verb,verb_compound,morph_case,log2_tag,unique_lemmas,log2_unique_lemmas,level,not_ann_words,ann_words,ann_unique_lemmas,olulisus
21249,valitsema,,in,3.044781,677,9.403012,n80,3776.0,1865.0,551.0,0.0
21223,toimuma,,in,2.873490,2200,11.103288,n80,24937.0,22095.0,1871.0,0.0
19354,treenima,,in,3.900242,142,7.149747,n80,387.0,433.0,124.0,0.0
19486,õpetama,,in,3.626783,219,7.774787,n80,788.0,630.0,181.0,0.0
19531,kasvama,üles,in,3.798366,172,7.426265,n80,391.0,320.0,158.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
18864,põgenema,,adit,4.285402,83,6.375039,n80,231.0,351.0,72.0,8.0e-05
19931,elutsema,,in,4.554589,74,6.209453,n80,150.0,94.0,70.0,8.0e-05
19940,müüma,,ill,4.610498,74,6.209453,n80,84.0,171.0,67.0,8.0e-05
19943,pöörduma,,adit,4.643856,74,6.209453,n80,416.0,325.0,63.0,8.0e-05


## json formaat

In [6]:
# initial example

messages = [{
            "role": "system",
            "content": (
                "Sa oled assistent, kes töötleb JSON-massiivi objekte, "
                "kus iga objekt sisaldab võtmeid 'lause' (täislause) ja 'clause' (märgitud fraas). "
                "Sinu ülesanne on otsustada, kas iga 'clause' viitab *asukohale* "
                "(näiteks linn, riik, piirkond, maamärk, park, aadress jne). \n\n"
                "Väljund peab olema JSON-massiiv, mis sisaldab ainult sõnu 'yes' või 'no', "
                "üks vastus iga sisendobjekti kohta, samas järjekorras nagu sisendis. "
                "Ei tohi lisada selgitusi, kirjavahemärke, tühikuid ega muud teksti. "
                "Näide:\n"
                "Sisend: [{\"lause\": \"Me läksime Pariisi\", \"clause\": \"Pariisi\"}, "
                "{\"lause\": \"Ta alustas tööd kell üheksa\", \"clause\": \"kell üheksa\"}]\n"
                "Väljund: [\"yes\", \"no\"]"
            ),
            }]
input_json = {
                "lause" : "",
                "clause" : ""
            }

## võtta spatial_obl tabelist näitelaused koos vajaliku infoga

In [7]:
# nt class_info tabelist rida ja sellele vastavad näited

#for i in range(len(filtered_class)):
#example_row = filtered_class.iloc[i]

example_row = filtered_class.iloc[4]

v = example_row["verb"]
v_c = example_row["verb_compound"]
m_c = example_row["morph_case"]

# transaction_head.form as head_form, lemma, spatial_obl.form as verb_form, verb, verb_compound, morph_case, sentence_id, sentence, phrase
query = f"""SELECT head_id, head_form, head_lemma, tbl2.form as verb_form, tbl1.verb, tbl1.verb_compound, 
            tbl1.morph_case, tbl1.sentence_id, tbl1.sentence, tbl2.phrase 

            FROM (
            SELECT head_id, form as head_form, lemma as head_lemma, verb, verb_compound, morph_case, 
            sentence_id, sentence, ekilex_tag 
            FROM spatial_obl 
            where 
            verb = '{v}' and 
            verb_compound='{v_c}' and 
            morph_case='{m_c}'
            and ekilex_tag = 'location'
            ) as tbl1

            join
            
            (SELECT * from transaction_head) as tbl2 on 
            tbl1.verb = tbl2.verb and 
            tbl1.verb_compound = tbl2.verb_compound and 
            tbl1.sentence_id = tbl2.sentence_id
            """

spatial_obl_ex = pd.read_sql(query, conn)
spatial_obl_ex
# kui tahta kõiki näiteid anda gpt-le
for j in range(len(spatial_obl_ex)):
    ex = spatial_obl_ex.iloc[j]
    peasona = ex["head_form"]
    phrase = ex["phrase"]
    input_json["lause"] = ex["sentence"]
    input_json["clause"] = peasona
    break

messages.append({"role":"user", "content":json.dumps(input_json, ensure_ascii=False)})   
    
spatial_obl_ex

,head_id,head_form,head_lemma,verb_form,verb,verb_compound,morph_case,sentence_id,sentence,phrase
0,54396,Aafrikas,Aafrika,kasvanud,kasvama,üles,in,31692,Aga Aafrikas üles kasvanud poisina ei teadnud ...,Aafrikas üles kasvanud
1,105869,põgenikelaagrites,põgenikelaager,kasvavad,kasvama,üles,in,61570,Kuni aga tõsimeeli taotleb kompromisslahendust...,mille jooksul kasvavad põgenikelaagrites üles ...
2,115977,USA-s,USA,kasvas,kasvama,üles,in,67436,"Meiegi ei saa olla ükskõiksed , eriti mitte Vä...",USA-s kasvas üles osa
3,137921,Tallinnas,Tallinn,kasvas,kasvama,üles,in,81015,I 1886 ) ja kasvas üles Tallinnas kasuisa Osca...,kasvas üles Tallinnas perekonnas
4,181685,Avinurmes,Avinurme,kasvanud,kasvama,üles,in,107978,"Olen Avinurmes üles kasvanud ja tean , et seal...",Olen Avinurmes üles kasvanud
...,...,...,...,...,...,...,...,...,...,...
315,27929765,Tartus,Tartu,kasvanud,kasvama,üles,in,18429554,Teistkordselt toob Liivimaalt pärit ja Tartus ...,Tartus üles kasvanud
316,28521921,Vändras,Vändra,kasvanud,kasvama,üles,in,18885996,"Marek Helm on Saaremaal sündinud , Vändras üle...",Vändras üles kasvanud
317,28701500,Eestis,Eesti,kasvanud,kasvama,üles,in,19018175,"Võin kinnitada , et vaatamata sellele , et väg...",ajal Eestis üles kasvanud
318,28742481,Eestis,Eesti,kasvanud,kasvama,üles,in,19046659,Aga kuidasmoodi te selgitate sellist olukorda ...,kes ei ole üles kasvanud Eestis


In [8]:
for m in messages:
    print(m, "\n")

{'role': 'system', 'content': 'Sa oled assistent, kes töötleb JSON-massiivi objekte, kus iga objekt sisaldab võtmeid \'lause\' (täislause) ja \'clause\' (märgitud fraas). Sinu ülesanne on otsustada, kas iga \'clause\' viitab *asukohale* (näiteks linn, riik, piirkond, maamärk, park, aadress jne). \n\nVäljund peab olema JSON-massiiv, mis sisaldab ainult sõnu \'yes\' või \'no\', üks vastus iga sisendobjekti kohta, samas järjekorras nagu sisendis. Ei tohi lisada selgitusi, kirjavahemärke, tühikuid ega muud teksti. Näide:\nSisend: [{"lause": "Me läksime Pariisi", "clause": "Pariisi"}, {"lause": "Ta alustas tööd kell üheksa", "clause": "kell üheksa"}]\nVäljund: ["yes", "no"]'} 

{'role': 'user', 'content': '{"lause": "Aga Aafrikas üles kasvanud poisina ei teadnud ta midagi Eestist , kus ta nüüd maailma väikseimat iseseisvat ortodoksset kirikut juhatab .", "clause": "Aafrikas"}'} 



### Siia kood, kuidas sööta messages gpt-le

In [9]:
conn.close()